In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

=========================================================
DADES (equivalent MATLAB)
=========================================================

In [ ]:
# motor
n_nom  = 1470      # rpm
Pot_nom = 1600     # W
mmot    = 6        # kg
Jmot    = 0.01     # kg·m2

In [ ]:
# reductor 40:1
i       = 40
mred    = 54
Jred    = 1.2
Jsor    = 2

In [ ]:
# sistema mecànic
Jtau    = 10
Npec    = 4
mpec    = 12
R       = 0.5

In [ ]:
# fregament
F1 = 100
F2 = 50   # F = F1 + F2*omega

=========================================================
PARÀMETRES EQUIVALENTS
=========================================================

In [ ]:
Jeq = Jmot + (Jsor + Jtau)/(i*i) + Npec*mpec*R*R/(i*i)
beq = F2 * R / (i*i)
M_r_eq = F1 * R / i
M_m = Pot_nom / (n_nom*np.pi/30)

=========================================================
MODELS MISO (G i H)
=========================================================

In [ ]:
num = [1]
num_neg = [-1]

In [ ]:
den_G = [Jeq, beq]
den_H = [Jeq, beq, 0]

In [ ]:
G1 = signal.lti(num, den_G)
G2 = signal.lti(num_neg, den_G)

In [ ]:
H1 = signal.lti(num, den_H)
H2 = signal.lti(num_neg, den_H)

=========================================================
TEMPS
=========================================================

In [ ]:
t_f = 20
dt = 1e-3
t = np.arange(0, t_f, dt)

=========================================================
ENTRADES
=========================================================

In [ ]:
M_m_ = M_m * np.ones_like(t)
M_r_eq_ = M_r_eq * np.ones_like(t)

=========================================================
RETARD (equivalent exp(-10s))
=========================================================

In [ ]:
def delay_signal(u, t, delay):
    dt = t[1] - t[0]
    shift = int(delay / dt)
    return np.concatenate((np.zeros(shift), u[:-shift]))

In [ ]:
M_m_delayed = delay_signal(M_m_, t, 10)

=========================================================
SIMULACIONS
=========================================================

In [ ]:
_, outG_1, _ = signal.lsim(G1, M_m_delayed, t)
_, outG_2, _ = signal.lsim(G2, M_r_eq_, t)

In [ ]:
_, outH_1, _ = signal.lsim(H1, M_m_delayed, t)
_, outH_2, _ = signal.lsim(H2, M_r_eq_, t)

In [ ]:
# superposició (MISO equivalent)
_, outG_T, _ = signal.lsim(G1, M_m_delayed - M_r_eq_, t)
_, outH_T, _ = signal.lsim(H1, M_m_delayed - M_r_eq_, t)

=========================================================
PLOTS
=========================================================

In [ ]:
plt.figure(figsize=(10,7))

In [ ]:
# --- motor
plt.subplot(3,2,1)
plt.plot(t, outG_1, linewidth=1.2)
plt.ylabel('velocitat [rad/s]')
plt.title('Resposta motor')
plt.grid()

In [ ]:
plt.subplot(3,2,2)
plt.plot(t, outH_1, linewidth=1.2)
plt.ylabel('gir [rad]')
plt.title('Resposta motor')
plt.grid()

In [ ]:
# --- fregament
plt.subplot(3,2,3)
plt.plot(t, outG_2, linewidth=1.2)
plt.ylabel('velocitat [rad/s]')
plt.title('Resposta fregament')
plt.grid()

In [ ]:
plt.subplot(3,2,4)
plt.plot(t, outH_2, linewidth=1.2)
plt.ylabel('gir [rad]')
plt.title('Resposta fregament')
plt.grid()

In [ ]:
# --- superposició velocitat
plt.subplot(3,2,5)
plt.plot(t, np.maximum(0, outG_1 + outG_2), linewidth=1.2)
plt.plot(t, outG_T, linewidth=1.2)
plt.legend(['Superposició', 'SISO equivalent'], frameon=False)
plt.ylabel('velocitat [rad/s]')
plt.xlabel('temps [s]')
plt.grid()

In [ ]:
# --- superposició gir
plt.subplot(3,2,6)
sumH = outH_1 + outH_2
plt.plot(t, sumH - np.min(sumH), linewidth=1.2)
plt.plot(t, outH_T, linewidth=1.2)
plt.legend(['Superposició', 'SISO equivalent'], frameon=False)
plt.ylabel('gir [rad]')
plt.xlabel('temps [s]')
plt.grid()

In [ ]:
plt.tight_layout()
plt.show()